# Full model grid on the 80/20 splitTrains all 60 configurations that have a folder in `metrics/` on the ARES 80/20 split(train 2770 / val 774, tested on the 22 external held-out pairs).**Before running:** upload `grid_bundle.zip` to the top level of your Google Drive.Build it on your machine with `python experiments/full_grid/make_bundle.py`.The runner is resumable: finished runs are skipped, and `--time-budget` stops it cleanly beforeColab cuts the session. Re-run the training cell as many times as needed.

## 1. GPU check

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csvimport torch; print("torch", torch.__version__, "| cuda", torch.cuda.is_available())

## 2. Mount Drive and unpack the project

In [ ]:
from google.colab import drivedrive.mount('/content/drive')import os, zipfile, shutilBUNDLE = '/content/drive/MyDrive/grid_bundle.zip'PROJECT = '/content/project'assert os.path.exists(BUNDLE), f'upload grid_bundle.zip to your Drive root first ({BUNDLE})'if not os.path.exists(PROJECT):    with zipfile.ZipFile(BUNDLE) as z: z.extractall('/content')print(sorted(os.listdir(PROJECT)))

## 3. Install the dependenciesColab already has torch, pandas, numpy and scikit-learn. These are the extras the encoders need.

In [ ]:
!pip -q install rdkit transformers chemprop torch-geometric gensim 2>&1 | tail -2print('done')

## 4. Results go to Drive, so nothing is lost when the session ends

In [ ]:
import osPROJECT = '/content/project'DRIVE_OUT = '/content/drive/MyDrive/grid_results'os.makedirs(DRIVE_OUT, exist_ok=True)# point the project's two output folders at Drive via symlinksfor rel, name in [('experiments/results/full_grid', 'full_grid'), ('checkpoints/full_grid', 'ckpt_full_grid')]:    local = os.path.join(PROJECT, rel)    target = os.path.join(DRIVE_OUT, name)    os.makedirs(target, exist_ok=True)    os.makedirs(os.path.dirname(local), exist_ok=True)    if os.path.islink(local) or os.path.exists(local):        if not os.path.islink(local): import shutil; shutil.rmtree(local)        else: os.remove(local)    os.symlink(target, local)print('results ->', DRIVE_OUT)

## 5. Build the split and list the grid

In [ ]:
%cd /content/project!python experiments/full_grid/prepare_data.py!python experiments/full_grid/run_grid.py --list | tail -5

## 6. TrainStart with the fixed-vector models: they are the fastest and tell you how long the rest will take.

In [ ]:
!python experiments/full_grid/run_grid.py --all --only fixed_vector --device cuda

Then everything else. `--time-budget 240` stops after 4 hours so the session is not killedmid-run; just run this cell again to continue where it left off.

In [ ]:
!python experiments/full_grid/run_grid.py --all --device cuda --time-budget 240

## 7. Collect and download

In [ ]:
!python experiments/full_grid/collect.py

In [ ]:
import shutilshutil.make_archive('/content/drive/MyDrive/grid_results_out', 'zip',                    '/content/drive/MyDrive/grid_results/full_grid')print('written to Drive: grid_results_out.zip — download it and unzip into')print('project/experiments/results/full_grid/ on your machine')